# 06 · Insights Summary & Tableau Extracts
### Credit Risk Analysis

**Notebook 6 of 6 — the wrap-up.** This notebook does three things:

1. **Consolidates** the headline numbers from the whole project into one place.
2. **Exports two flat CSVs** that the Tableau dashboard connects to — a
   client-level file (for segment/KPI charts) and a tidy payment-behavior file
   (for the time-based charts).
3. **Writes a one-page findings summary** to `docs/` — the plain-language
   deliverable a hiring manager or stakeholder reads first.

Nothing new is modelled here; it turns the analysis into shareable artifacts.

## 1. Setup and load both sources

**What:** Load the labelled client table (from notebook 02) and the raw monthly
`credit_record` (for the time-based extract and behavioral KPIs).

**Why:** The two Tableau extracts live at different grains — one row per client,
one row per client-month — so we need both sources. Re-using the saved labelled
table keeps everything consistent with the target we engineered.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../data/processed/credit_risk_labelled.csv")
credit = pd.read_csv("../data/raw/credit_record.csv")
print(f"Labelled clients: {len(df):,} | credit-record rows: {len(credit):,}")

Labelled clients: 36,457 | credit-record rows: 1,048,575


## 2. Build the client-level features for Tableau

**What:** Recreate the decoded, business-readable features and bands used in the
EDA (age, income quintile, employment length, etc.) and a readable
`target_label` ("Good" / "Bad").

**Why:** Tableau visualizes flat tables, not Python — so every grouping a chart
might need (age band, income band, Good/Bad label) must exist as a column in the
extract. Pre-computing them here keeps the Tableau workbook simple: drag-and-drop,
no calculated fields required.

In [2]:
c = df.copy()
c["age_years"] = (-c["DAYS_BIRTH"] / 365.25).round(1)
c["age_band"] = pd.cut(c["age_years"], bins=[0,25,35,45,55,65,200],
                       labels=["<25","25-34","35-44","45-54","55-64","65+"])
c["is_employed"] = (c["DAYS_EMPLOYED"] < 0).astype(int)
c["employ_years"] = np.where(c["is_employed"]==1, -c["DAYS_EMPLOYED"]/365.25, 0).round(1)
c["employ_band"] = pd.cut(c["employ_years"], bins=[-0.01,1,3,5,10,100],
                          labels=["<1 yr","1-3 yr","3-5 yr","5-10 yr","10+ yr"])
c["employ_band"] = c["employ_band"].cat.add_categories("Not employed")
c.loc[c["is_employed"]==0, "employ_band"] = "Not employed"
c["income_band"] = pd.qcut(c["AMT_INCOME_TOTAL"], 5,
                           labels=["Q1 (lowest)","Q2","Q3","Q4","Q5 (highest)"], duplicates="drop")
c["target_label"] = np.where(c["target"]==1, "Bad", "Good")

client_cols = ["ID","target","target_label","age_years","age_band",
               "AMT_INCOME_TOTAL","income_band","employ_years","employ_band","is_employed",
               "CODE_GENDER","FLAG_OWN_CAR","FLAG_OWN_REALTY","NAME_INCOME_TYPE",
               "NAME_EDUCATION_TYPE","NAME_FAMILY_STATUS","NAME_HOUSING_TYPE",
               "OCCUPATION_TYPE","CNT_CHILDREN","CNT_FAM_MEMBERS"]
client_level = c[client_cols].copy()
client_level["OCCUPATION_TYPE"] = client_level["OCCUPATION_TYPE"].fillna("Unknown")
print("client_level shape:", client_level.shape)
client_level.head(3)

client_level shape: (36457, 20)


,ID,target,target_label,age_years,age_band,AMT_INCOME_TOTAL,income_band,employ_years,employ_band,is_employed,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,OCCUPATION_TYPE,CNT_CHILDREN,CNT_FAM_MEMBERS
0,5008804,0,Good,32.9,25-34,427500.0,Q5 (highest),12.4,10+ yr,1,M,Y,Y,Working,Higher education,Civil marriage,Rented apartment,Unknown,0,2.0
1,5008805,0,Good,32.9,25-34,427500.0,Q5 (highest),12.4,10+ yr,1,M,Y,Y,Working,Higher education,Civil marriage,Rented apartment,Unknown,0,2.0
2,5008806,0,Good,58.8,55-64,112500.0,Q1 (lowest),3.1,3-5 yr,1,M,Y,Y,Working,Secondary / secondary special,Married,House / apartment,Security staff,0,2.0


In [3]:
client_path = "../tableau/extracts/client_level.csv"
client_level.to_csv(client_path, index=False)
print(f"Exported -> {client_path}  ({len(client_level):,} rows, {client_level.shape[1]} cols)")

Exported -> ../tableau/extracts/client_level.csv  (36,457 rows, 20 cols)


## 3. Build the payment-behavior extract for Tableau

**What:** From `credit_record`, derive each record's days-past-due tier, account
age (months on book), and a readable status group, then export the tidy
client-month table.

**Why:** This drives the *time-based* dashboard views (vintage curve, status mix
over account age). Keeping it tidy — one row per client-month with a few derived
columns — lets Tableau build trend charts directly by aggregating over
`months_on_book`.

In [4]:
dpd_tier = {"X":0,"C":0,"0":0,"1":30,"2":60,"3":90,"4":120,"5":150}
credit["dpd"] = credit["STATUS"].map(dpd_tier)
credit["months_on_book"] = credit["MONTHS_BALANCE"] - credit.groupby("ID")["MONTHS_BALANCE"].transform("min")
credit["status_group"] = np.select(
    [credit["STATUS"].isin(["C","X"]), credit["STATUS"]=="0", credit["STATUS"]=="1"],
    ["Paid / no loan","1-29 DPD","30-59 DPD"], default="60+ DPD")
credit["is_60plus"] = (credit["dpd"] >= 60).astype(int)

pay_cols = ["ID","MONTHS_BALANCE","months_on_book","STATUS","status_group","dpd","is_60plus"]
payment_behavior = credit[pay_cols].copy()
pay_path = "../tableau/extracts/payment_behavior.csv"
payment_behavior.to_csv(pay_path, index=False)
print(f"Exported -> {pay_path}  ({len(payment_behavior):,} rows, {payment_behavior.shape[1]} cols)")

Exported -> ../tableau/extracts/payment_behavior.csv  (1,048,575 rows, 7 cols)


## 4. Consolidated KPIs

**What:** Compute the project's headline numbers in one cell — dataset size,
overall bad rate, the strongest segment gaps, and the behavioral signals from
notebook 04.

**Why:** These are the figures for the dashboard's KPI cards and the findings
summary. Computing them in one place (rather than copying from earlier notebooks)
means the summary can never drift from the data.

In [5]:
n_clients = len(df)
overall_bad = df["target"].mean()*100

# Segment gaps
def rate_by(col):
    g = client_level.groupby(col, observed=True)["target"].agg(n="size", bad="mean")
    g = g[g["n"] >= 100]; g["bad_%"] = g["bad"]*100
    return g.sort_values("bad_%", ascending=False)

age_gap = rate_by("age_band")
emp_gap = rate_by("employ_band")

# Behavioral: time to first 60+ and early-warning lift (from credit_record)
bad_ids = credit.groupby("ID")["dpd"].max()
bad_ids = bad_ids[bad_ids >= 60].index
first_bad = credit[(credit["ID"].isin(bad_ids)) & (credit["dpd"]>=60)].groupby("ID")["months_on_book"].min()
within24 = (first_bad <= 24).mean()*100

W = 6
obs = credit.groupby("ID")["months_on_book"].max()
elig = obs[obs >= W].index
sub = credit[credit["ID"].isin(elig)]
early = sub[sub["months_on_book"] < W].groupby("ID")["dpd"].max() >= 30
later = sub[sub["months_on_book"] >= W].groupby("ID")["dpd"].max() >= 60
fl = pd.concat([early.rename("e"), later.rename("l")], axis=1).dropna()
r = fl.groupby("e")["l"].mean()*100
ew_lift = r[True]/r[False]

kpis = {
    "clients_analyzed": n_clients,
    "overall_bad_rate_%": round(overall_bad, 2),
    "highest_risk_age": (age_gap.index[0], round(age_gap['bad_%'].iloc[0],2)),
    "lowest_risk_age": (age_gap.index[-1], round(age_gap['bad_%'].iloc[-1],2)),
    "median_months_to_default": int(first_bad.median()),
    "defaults_within_24m_%": round(within24,1),
    "early_warning_lift_x": round(ew_lift,1),
}
for k,v in kpis.items(): print(f"{k:28s}: {v}")

clients_analyzed            : 36457
overall_bad_rate_%          : 1.69
highest_risk_age            : ('65+', np.float64(2.54))
lowest_risk_age             : ('35-44', np.float64(1.42))
median_months_to_default    : 8
defaults_within_24m_%       : 91.5
early_warning_lift_x        : 11.7


## 5. Write the one-page findings summary

**What:** Generate `docs/findings_summary.md` — a concise, plain-language summary
of the business problem, method, and the five headline findings, with the numbers
filled in from the KPI cell above.

**Why:** This is the artifact a busy reader opens first. Generating it from the
computed KPIs (rather than hand-typing) guarantees it always matches the analysis.

In [6]:
summary = '''# Credit Risk Analysis — One-Page Findings Summary

## The question
Using a credit-card application dataset, identify **which applicants are most
likely to default** (defined as ever reaching 60+ days past due) and **what drives
that risk** — the evidence a lender uses to shape approval and monitoring.

## What we did
- The dataset has **no default label**, so we engineered one from monthly
  repayment history (**bad = ever 60+ days past due**), validated against a
  24-month vintage window.
- Joined the label to applicant features, giving **{n:,} analyzable clients** with
  an overall bad rate of **{ob:.2f}%** (a heavily imbalanced problem).
- Analyzed risk by applicant segment, studied payment behavior over time, and fit
  an interpretable logistic model to isolate each factor's effect.

## Five headline findings
1. **Default is rare and imbalanced** — only **{ob:.2f}%** of clients go bad, so
   the analysis uses bad-*rate* comparisons and rank/recall metrics, not accuracy.
2. **Risk varies most by life-stage and stability factors** — e.g. age band risk
   ranges from **{la_r:.2f}%** ({la_s}) up to **{ha_r:.2f}%** ({ha_s}).
3. **Bad clients reveal themselves quickly** — the median time to first serious
   delinquency is **{mm} months**, and **{w24:.0f}%** of eventual defaults occur
   within 24 months of account opening.
4. **Early lateness is a strong early-warning signal** — clients who slip (30+ DPD)
   in their first six months are about **{ew:.1f}x** more likely to seriously
   default later than clients with a clean start.
5. **Applicant attributes carry limited predictive power** — the model beats its
   baseline but only modestly, because the strongest predictors (utilization,
   prior delinquency, debt-to-income) are not in the application data. The value
   is in *explaining* risk drivers, not in a high accuracy score.

## How to use this repo
Run notebooks `01`–`06` in order; the Tableau dashboard connects to the two CSVs
in `tableau/extracts/`. Full methodology is documented inside each notebook.

*Generated automatically from the analysis — figures match the latest run.*
'''.format(
    n=kpis["clients_analyzed"], ob=kpis["overall_bad_rate_%"],
    ha_s=kpis["highest_risk_age"][0], ha_r=kpis["highest_risk_age"][1],
    la_s=kpis["lowest_risk_age"][0], la_r=kpis["lowest_risk_age"][1],
    mm=kpis["median_months_to_default"], w24=kpis["defaults_within_24m_%"],
    ew=kpis["early_warning_lift_x"])

with open("../docs/findings_summary.md", "w", encoding="utf-8") as f:
    f.write(summary)
print("Wrote ../docs/findings_summary.md")
print(summary[:600])

Wrote ../docs/findings_summary.md
# Credit Risk Analysis — One-Page Findings Summary

## The question
Using a credit-card application dataset, identify **which applicants are most
likely to default** (defined as ever reaching 60+ days past due) and **what drives
that risk** — the evidence a lender uses to shape approval and monitoring.

## What we did
- The dataset has **no default label**, so we engineered one from monthly
  repayment history (**bad = ever 60+ days past due**), validated against a
  24-month vintage window.
- Joined the label to applicant features, giving **36,457 analyzable clients** with
  an overall bad ra


## 6. Wrap-up — building the Tableau dashboard

The two extracts are ready in `tableau/extracts/`:

- **`client_level.csv`** — one row per client. Use for KPI cards (overall bad
  rate, client count) and segment bar charts (bad rate by age band, income
  quintile, housing, employment length). Put `target` on a measure as
  `AVG(target)` to get the bad rate.
- **`payment_behavior.csv`** — one row per client-month. Use for the vintage curve
  (`AVG(is_60plus)` by `months_on_book`) and the status-composition area chart
  (share of `status_group` by `months_on_book`).

**Next steps (in Tableau Public Desktop):** connect to each CSV, build the
worksheets above, assemble them into a dashboard, save the `.twbx` into
`tableau/workbooks/`, publish to Tableau Public, and paste the link into the
project README and `tableau/README.md`.

**Project complete.** Six documented notebooks take the data from raw files to an
engineered target, segment and behavioral insight, an interpretable model, and
dashboard-ready extracts — with every decision written down.